# Impact of Prompting Strategies on Information Extraction from AI News Articles Using Open-Source LLMs

## Part 1: Data Preparation and Exploratory Data Analysis (EDA)

In [1]:
import pandas as pd

# Import dataset
df = pd.read_csv("MIT_AI_ARTICLES.csv")

print("Original article number:", len(df))

# Hold on only necessary columns
df = df[[
    "title",
    "publication_date",
    "summary",
    "body",
    "url"
]]

# Remove the ones with an empty Body.
df = df.dropna(subset=["body"])

# Calculate word count
df["word_count"] = df["body"].astype(str).str.split().str.len()

# Remove shorter or longer articles
clean_df = df[
    (df["word_count"] >= 200) &
    (df["word_count"] <= 3000)
]

print("Number of cleaned articles:", len(clean_df))

# Save
clean_df.to_csv("clean_articles.csv", index=False)

clean_df.head()

Original article number: 314
Number of cleaned articles: 314


,title,publication_date,summary,body,url,word_count
0,A new model predicts how molecules will dissol...,"August 19, 2025",Solubility predictions could make it easier to...,"Using machine learning, MIT chemical engineers...",https://news.mit.edu/2025/new-model-predicts-h...,1149
1,Researchers glimpse the inner workings of prot...,"August 18, 2025",A new approach can reveal the features AI mode...,"Within the past few years, models that can pre...",https://news.mit.edu/2025/researchers-glimpse-...,1025
2,How AI could speed the development of RNA vacc...,"August 15, 2025",MIT engineers used a machine-learning model to...,"Using artificial intelligence, MIT researchers...",https://news.mit.edu/2025/how-ai-could-speed-d...,1006
3,"Using generative AI, researchers design compou...","August 14, 2025",The team used two different AI approaches to d...,"With help from artificial intelligence, MIT re...",https://news.mit.edu/2025/using-generative-ai-...,1076
4,A new way to test how well AI systems classify...,"August 13, 2025",As large language models increasingly dominate...,Is this movie review a rave or a pan? Is this ...,https://news.mit.edu/2025/new-way-test-how-wel...,1186


In [2]:
clean_df["word_count"].describe()

count     314.000000
mean     1053.404459
std       283.800000
min       259.000000
25%       910.750000
50%      1051.500000
75%      1172.000000
max      2304.000000
Name: word_count, dtype: float64

After removing missing values and filtering articles outside the 200–3000 word range, all 314 articles remained in the dataset. 
Therefore, no additional article removal was required.

#### Control of numbers company, model and tech at all dataset

In [3]:
import re
from collections import Counter

companies = [
    "OpenAI",
    "Google",
    "Meta",
    "Microsoft",
    "Anthropic",
    "Nvidia",
    "Amazon",
    "IBM",
    "Adobe",
    "Tesla",
    "Apple",
    "DeepMind"
]

all_text = " ".join(clean_df["body"].astype(str))

for company in companies:
    count = len(re.findall(company, all_text, flags=re.IGNORECASE))
    print(f"{company}: {count}")

OpenAI: 28
Google: 79
Meta: 108
Microsoft: 14
Anthropic: 6


Nvidia: 21
Amazon: 34
IBM: 107
Adobe: 8
Tesla: 4
Apple: 17
DeepMind: 9


In [4]:
models = [
    "GPT",
    "Claude",
    "Gemini",
    "Llama",
    "Mistral"
]

for model in models:
    print(model, all_text.count(model))

GPT 115
Claude 13
Gemini 5
Llama 4
Mistral 3


In [5]:
techs = [
    "Generative AI",
    "Large Language Model",
    "Machine Learning",
    "Deep Learning",
    "Reinforcement Learning",
    "Neural Network",
    "Computer Vision",
    "Multimodal"
]

for tech in techs:
    print(tech, all_text.count(tech))

Generative AI 24
Large Language Model 0
Machine Learning 42
Deep Learning 2
Reinforcement Learning 0
Neural Network 0
Computer Vision 15
Multimodal 2


## Part 2 : Selecting 100 articles

In [6]:
keywords = [
    "OpenAI",
    "Google",
    "Meta",
    "Microsoft",
    "Anthropic",
    "Nvidia",
    "GPT",
    "Claude",
    "Gemini",
    "Llama"
]

mask = clean_df["body"].astype(str).apply(
    lambda x: any(k.lower() in x.lower() for k in keywords)
)
entity_articles = clean_df[mask]

eval_df = entity_articles.sample(n=100, random_state=42).reset_index(drop=True)

eval_df.insert(0, "article_id", [f"art_{i:03d}" for i in range(len(eval_df))])

eval_df.to_csv("evaluation_articles.csv", index=False)

print("Number of articles:", len(eval_df))
eval_df[["article_id", "title", "word_count"]].head()

Number of articles: 100


,article_id,title,word_count
0,art_000,Stratospheric safety standards: How aviation c...,1513
1,art_001,Can robots learn from machine dreams?,1059
2,art_002,School of Engineering welcomes new faculty,1564
3,art_003,Envisioning a future where health care tech le...,1084
4,art_004,Introducing the MIT Generative AI Impact Conso...,1652


## Part 3 — Building the Annotation Sheet (Ground Truth)

In [7]:
CANDIDATE_COMPANIES = [
    "OpenAI", "Anthropic", "Google DeepMind", "Google", "DeepMind",
    "Meta", "Microsoft", "Nvidia", "Amazon", "IBM", "Adobe", "Tesla", "Apple",
]
CANDIDATE_MODELS = [
    "GPT-5", "GPT-4", "GPT", "Claude", "Gemini", "Llama", "Mistral",
]
TECHNOLOGY_LABELS = [
    "Retrieval-Augmented Generation", "RAG",
    "Reinforcement Learning",
    "Generative AI",
    "Multimodal AI",
    "AI Agents",
]

def find_mentions(text, candidates):
    text_lower = str(text).lower()
    found = []
    for candidate in candidates:
        if candidate.lower() in text_lower:
            found.append(candidate)
    return found

annotation_rows = []
for _, row in eval_df.iterrows():
    suggested_companies = find_mentions(row["body"], CANDIDATE_COMPANIES)
    suggested_models = find_mentions(row["body"], CANDIDATE_MODELS)
    suggested_technologies = find_mentions(row["body"], TECHNOLOGY_LABELS)

    annotation_rows.append({
        "article_id": row["article_id"],
        "title": row["title"],
        "url": row["url"],
        "body": row["body"],
        # Pre-filled guesses 
        "suggested_companies": "; ".join(suggested_companies),
        "suggested_models": "; ".join(suggested_models),
        "suggested_technologies": "; ".join(suggested_technologies),
        # I fill these in by hand with the CORRECT, final entity list.
        "gt_companies": "",
        "gt_models": "",
        "gt_technologies": "",
    })

annotation_df = pd.DataFrame(annotation_rows)
annotation_df.to_csv("annotation_sheet.csv", index=False)

print("annotation_sheet.csv created:", len(annotation_df), "rows")

annotation_df.drop(columns=["body"]).head()

annotation_sheet.csv created: 100 rows


,article_id,title,url,suggested_companies,suggested_models,suggested_technologies,gt_companies,gt_models,gt_technologies
0,art_000,Stratospheric safety standards: How aviation c...,https://news.mit.edu/2024/stratospheric-safety...,Microsoft,,RAG,,,
1,art_001,Can robots learn from machine dreams?,https://news.mit.edu/2024/can-robots-learn-mac...,Amazon,GPT,Generative AI,,,
2,art_002,School of Engineering welcomes new faculty,https://news.mit.edu/2024/school-engineering-w...,Google; DeepMind; Meta,,RAG; Reinforcement Learning,,,
3,art_003,Envisioning a future where health care tech le...,https://news.mit.edu/2025/envisioning-future-w...,Meta,,RAG,,,
4,art_004,Introducing the MIT Generative AI Impact Conso...,https://news.mit.edu/2025/introducing-mit-gene...,OpenAI,GPT,RAG; Generative AI; AI Agents,,,


## Part 4: Prompt Design

Three prompting strategies were implemented in `prompt_templates.py`:
- **Zero-shot**: task description only.
- **Few-shot**: task description plus two annotated example articles.
- **Structured**: task description plus an explicit JSON output schema.

AI technology extraction is restricted to the five predefined labels used during ground truth annotation. Companies and models remain open-set.

In [8]:
from prompt_templates import build_prompt, PROMPT_BUILDERS

# Use a real evaluation article to preview the prompts
eval_df = pd.read_csv("evaluation_articles.csv")
sample_article = eval_df.loc[eval_df["article_id"] == "art_004", "body"].values[0]

print(build_prompt("zero_shot", sample_article)[:800])

Extract all company names, AI model names, and AI technologies mentioned in the article.
Only consider the following AI technologies (use these exact labels, and only include a technology if it is explicitly mentioned in the article): Retrieval-Augmented Generation (RAG), Reinforcement Learning, Generative AI, Multimodal AI, AI Agents.

Report the results as three labeled lists, in this exact format:
Companies:
- ...
AI Models:
- ...
AI Technologies:
- ...

If a category has no entities, leave its list empty.

Article:
From crafting complex code to revolutionizing the hiring process, generative artificial intelligence is reshaping industries faster than ever before — pushing the boundaries of creativity, productivity, and collaboration across countless domains.
Enter theMIT Generative AI I


In [9]:
print(build_prompt("few_shot", sample_article)[:1200])

Extract all company names, AI model names, and AI technologies mentioned in the article.
Only consider the following AI technologies (use these exact labels, and only include a technology if it is explicitly mentioned in the article): Retrieval-Augmented Generation (RAG), Reinforcement Learning, Generative AI, Multimodal AI, AI Agents.

Report the results as three labeled lists, in the same format as the examples below. If a category has no entities, leave its list empty.

Example 1
Article:
OpenAI announced GPT-5 while introducing new Retrieval-Augmented Generation capabilities. Anthropic also reported improvements to Claude.

Output:
Companies:
- OpenAI
- Anthropic
AI Models:
- GPT-5
- Claude
AI Technologies:
- Retrieval-Augmented Generation (RAG)

Example 2
Article:
Google DeepMind unveiled Gemini's new multimodal reasoning capabilities, allowing the model to process images, audio, and text together. Meta's research team separately published work on reinforcement learning agents tha

In [10]:
print(build_prompt("structured", sample_article)[:800])

Extract all company names, AI model names, and AI technologies mentioned in the article.
Only consider the following AI technologies (use these exact labels, and only include a technology if it is explicitly mentioned in the article): Retrieval-Augmented Generation (RAG), Reinforcement Learning, Generative AI, Multimodal AI, AI Agents.

Return ONLY a JSON object with this exact structure, and nothing else (no explanations, no markdown code fences):
{
  "companies": [],
  "models": [],
  "technologies": []
}

If a category has no entities, return an empty list for it.

Article:
From crafting complex code to revolutionizing the hiring process, generative artificial intelligence is reshaping industries faster than ever before — pushing the boundaries of creativity, productivity, and collabora


All three prompt builders take an article body and return a ready-to-send prompt string. These will be used in Part 5 (Model Inference) to query Llama 3 and Gemma 3 across the six model x strategy combinations.

## Part 5: Model Inference

Inference was run separately via `run_inference.py` rather than inline in this notebook, since it issues real API calls (with retries and incremental checkpointing) and takes several minutes to complete.

**Setup**
- Provider: OpenRouter (OpenAI-compatible API)
- Models: `meta-llama/llama-3.1-8b-instruct` (closest currently-available match to "Llama 3", since the original Llama 3 release has been deprecated on OpenRouter) and `google/gemma-3-12b-it`
- 100 evaluation articles x 3 prompting strategies x 2 models = 600 API calls
- `temperature=0.0` for deterministic, reproducible extractions

Command used:
```
python run_inference.py
```

The script is resumable: it skips any (article_id, model, strategy) triple already present in `raw_extractions.csv`, so an interrupted run can simply be re-launched without repeating (and re-paying for) completed calls.

In [11]:
raw_df = pd.read_csv("raw_extractions.csv")

print("Total rows:", len(raw_df))
print("Unique articles:", raw_df["article_id"].nunique())
print("Rows with an error:", raw_df["error"].notna().sum())
print()
print(raw_df.groupby(["model", "strategy"]).size())

Total rows: 600
Unique articles: 100
Rows with an error: 0

model   strategy  
gemma3  few_shot      100
        structured    100
        zero_shot     100
llama3  few_shot      100
        structured    100
        zero_shot     100
dtype: int64


All 600 combinations completed successfully with no errors and no empty responses.

In [12]:
# Preview one structured-prompting output
sample = raw_df[
    (raw_df["article_id"] == "art_004")
    & (raw_df["model"] == "gemma3")
    & (raw_df["strategy"] == "structured")
].iloc[0]

print(sample["raw_output"])

```json
{
  "companies": [
    "Analog Devices",
    "The Coca-Cola Co.",
    "OpenAI",
    "Tata Group",
    "SK Telecom",
    "TWG Global",
    "Tata Sons Ltd."
  ],
  "models": [
    "ChatGPT",
    "UNIVAC"
  ],
  "technologies": [
    "Generative AI",
    "Retrieval-Augmented Generation (RAG)",
    "Reinforcement Learning",
    "Multimodal AI",
    "AI Agents"
  ]
}
```


**Observation.** Manual inspection of a few outputs during a smoke test (3 articles) surfaced a recurring failure pattern: for `art_002`, an article about new faculty hires, Gemma 3 listed universities (MIT, Stanford, Harvard, ...) as "companies", and under the structured prompt it returned all five predefined technology labels even though most were not explicitly mentioned in the article. This kind of over-generation / hallucination is exactly what Part 8 (Error Analysis) will quantify and compare across models and prompting strategies — it was left as-is rather than patched, since it is part of the behavior being measured.